# Spanner walkthrough

Fine-tune BERT on the kangaroo QA set, then ask it questions about `data/kangaroo.txt`.

Run from the repository root after `pip install -e .`

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from spanner.data import load_examples, read_text
from spanner.evaluate import evaluate
from spanner.predict import QuestionAnswerer
from spanner.train import train

## The data

Each question's answer is an exact span of its context passage.

In [ ]:
examples = load_examples("data/kangaroo_qa.json")
for e in examples[:3]:
    print(f"Q: {e.question}\nA: {e.answer}\n")

## Baseline: a SQuAD 2.0 model with no domain fine-tuning

In [ ]:
baseline = QuestionAnswerer("deepset/bert-base-uncased-squad2")
evaluate(baseline, examples)

## Fine-tune

In [ ]:
model_dir = train("data/kangaroo_qa.json", "models/spanner-kangaroo", epochs=3)

In [ ]:
qa = QuestionAnswerer(str(model_dir))
evaluate(qa, examples)

## Ask questions about a new document

In [ ]:
context = read_text("data/kangaroo.txt")

for question in [
    "What do kangaroos eat?",
    "Where do female kangaroos nurture their young?",
    "Which kangaroo is known for its distinctive appearance?",
]:
    answer = qa.answer(question, context)
    print(f"{question}\n  -> {answer.text}  ({answer.score:.2f})")